# Step 01 — SQLite post-fit pipeline and hidden-mechanism summaries

This notebook reruns the **legacy single-current post-fit summaries** used for reviewer-response context.

## Reuse from `astro_atf_analysis_improved_sectioned.ipynb`

None of the ATF parsing, preprocessing, or sweep-feature extraction logic is reused here.  
This step interprets the **historical Optuna DB summaries**, not the new ATF dataset.

## Interpretation boundary

Because historical control provenance remains unresolved in step 00, the outputs below should be treated as:
- structural-confounding diagnostics,
- effective-parameter summaries,
- and provisional mechanism narratives,

not as final reviewer-facing mechanism proof.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.postfit_sqlite import d_pk_invariance_check, run_step01_postfit_sqlite

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT

In [ ]:
results = run_step01_postfit_sqlite(PROJECT_ROOT)
top_trials_df = results["top_trials_all_dbs"]
effective_df = results["effective_parameter_summary"]
representative_df = results["representative_mechanism_summary"]

print("written outputs:", sorted((PROJECT_ROOT / "outputs" / "postfit_sqlite").glob("*.csv")))
print("top-trial rows:", len(top_trials_df), "effective rows:", len(effective_df), "representative rows:", len(representative_df))

## `d × pk` confounding check

In [ ]:
base_params = {
    "gki": 50.0,
    "pk": 2e-4,
    "d": 3.0,
    "gt": 8.0,
    "gs": 10.0,
    "zth": 0.2,
    "zs": 0.05,
    "K_bath_value_middle": 8.2,
    "eps": 0.01,
    "eps_middle": 0.5,
    "wo": 1500.0,
    "ca": 400.0,
    "gl_a": 0.01,
    "Va_l": -70.0,
    "Va_s": -90.0,
    "switching_function": "sigmoid",
}
check = d_pk_invariance_check(base_params)
display(
    pd.DataFrame(
        [
            {
                "P_gap_eff_a": check.P_gap_eff_a,
                "P_gap_eff_b": check.P_gap_eff_b,
                "I_kgap_a": check.I_kgap_a,
                "I_kgap_b": check.I_kgap_b,
                "max_abs_rhs_delta": float(abs(check.dzdt_a - check.dzdt_b).max()),
            }
        ]
    )
)

## Effective parameter summary (all 18 DBs)

In [ ]:
display(effective_df)

fig, ax = plt.subplots(figsize=(10, 4))
for condition, group in effective_df.groupby("condition"):
    ax.plot(group["current_na"], group["P_gap_eff"], marker="o", label=condition)
ax.set_xlabel("Current (nA)")
ax.set_ylabel("P_gap_eff")
ax.set_title("Effective gap permeability across historical best trials")
ax.legend()
plt.show()

## Representative mechanism summary

In [ ]:
display(representative_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(representative_df["condition"], representative_df["gap_to_kir_integral_ratio"])
ax.set_ylabel("gap_to_kir_integral_ratio")
ax.set_title("Representative mechanism contrast by condition")
plt.show()

## Consequence for the reviewer response

Step 01 still supports the mechanistic framing that **effective combinations** matter more than raw `d` or `pk` alone.  
However, because it is built on legacy single-current fits — and because historical control provenance is still unresolved — the strongest reviewer-facing evidence among steps 00-02 remains the ATF-based step 02 thresholds and region summaries.